# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
print(cat.loc[cat['product_name'].str.lower().str.contains('pop')])
print(len(cat.loc[cat['product_name'].str.lower().str.contains('pop')]))

             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast
3


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = cat.loc[cat['product_name'].str.lower().str.contains('pop')]
print(strawberry)


             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


**Pop-Tart rows that are a different product:**
There are no pop tart rows that are a different flavor, though other flavors may include something like blueberry.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW'
}
cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])

print(cat[['sku', 'product_name', 'canonical_sku']])

             sku          product_name  canonical_sku
0          PT-12  pop-tarts strawberry  POPTART-STRAW
1     SKU#459812   Strawberry Pop-Tart  POPTART-STRAW
2  POPTART-STRAW  Pop-Tarts Strawberry  POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print(cat['category'].value_counts())
cat['category'] = cat['category'].str.strip().str.title()
print(cat['category'].value_counts())


category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
cat['normalized_name'] = (cat['product_name'].str.lower().str.replace(r'[^\w\s]', '', regex=True).str.strip().apply(lambda name: ' '.join(sorted(name.split()))))

groups = cat.groupby('normalized_name').agg(sku_count=('sku', 'nunique'),skus=('sku', lambda x: list(x)), names=('product_name', lambda x: list(x)))
print(groups.loc[groups['sku_count'] > 1])

                     sku_count                    skus  \
normalized_name                                          
poptarts strawberry          2  [PT-12, POPTART-STRAW]   

                                                            names  
normalized_name                                                    
poptarts strawberry  [pop-tarts strawberry, Pop-Tarts Strawberry]  


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
assert cat['canonical_sku'].notna().all()
assert len(cat['canonical_sku']) == len(cat)

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) I would probably choose the category of Grocery which contains the product bottled water. Since bottled water contains two SKUs, I could group them by number of bottles in a package. I would also choose flashlights and would group them by brand or strength of the flashlight.

b) Consolidation stops being a data-cleaning task when we are not sure if two products are the same type and if they should be grouped together. I would probably ask whoever is in charge of inventory or a store manager if I am able to reach them. I could ask a data scientist or business consultant within Walmart if they know the particular dataset well.